# E04 Matrix Factorization Initialization Ablations

This experiment keeps the MatrixFactorization objective from E02 but changes only the factor initialization.

The optimized variables are $L,R \in \mathbb{R}^{d \times r_f}$ and the loss is

$$
f(L,R) = \frac{1}{2d^2}\lVert LR^\top - X^\star \rVert_F^2.
$$

The goal is to make initialization harder and more ill-conditioned: tiny factors, oversized factors, left/right scale imbalance, and column-wise factor condition numbers up to $10^4$.

## Setup

In [ ]:
import os
import pathlib
import sys
import time
for name in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(name, "1")

import joblib
import IPython.display
import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT = pathlib.Path.cwd().resolve()
if not (PROJECT / "problems").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

import optimizers
import plotting
import problems.MatrixConstruction
import problems.MatrixFactorization
import util

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print(f"project = {PROJECT}")
print(f"torch   = {torch.__version__}")
print(f"joblib  = {joblib.__version__}")


## Parameters And Grid Preview

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_NAME = "float64"
ALGOS = ["Muon", "Muon-Exact", "Shampoo", "Adam", "SGD"]
SEEDS = list(range(10))
SMOKE_TEST = False
SMOKE_TEST_MAX_STEPS = 10
FULL_ITERS = 2000

BASE_SPEC = dict(
    d=60,
    rank=5,
    factor_rank=5,
    lr=0.01,
    spectrum="hard-cutoff",
    kappa=1.0,
    iters=SMOKE_TEST_MAX_STEPS if SMOKE_TEST else FULL_ITERS,
    early_stop=True,
    early_stop_min_steps=200,
    early_stop_patience=200,
    early_stop_min_delta=1e-4,
    device_type=DEVICE.type,
    dtype_name=DTYPE_NAME,
)
INIT_SCENARIOS = [
    dict(scenario="baseline-small", init_mode="gaussian", left_scale=0.01, right_scale=0.01, init_kappa=1.0),
    dict(scenario="tiny-flat", init_mode="gaussian", left_scale=1e-4, right_scale=1e-4, init_kappa=1.0),
    dict(scenario="large-overshoot", init_mode="gaussian", left_scale=0.5, right_scale=0.5, init_kappa=1.0),
    dict(scenario="left-tiny-right-large", init_mode="gaussian", left_scale=1e-4, right_scale=1.0, init_kappa=1.0),
    dict(scenario="left-large-right-tiny", init_mode="gaussian", left_scale=1.0, right_scale=1e-4, init_kappa=1.0),
    dict(scenario="illcond-columns-100", init_mode="column-ill-conditioned", left_scale=0.05, right_scale=0.05, init_kappa=100.0),
    dict(scenario="illcond-columns-10000", init_mode="column-ill-conditioned", left_scale=0.05, right_scale=0.05, init_kappa=10000.0),
    dict(scenario="opposite-illcond-10000", init_mode="opposite-column-ill-conditioned", left_scale=0.05, right_scale=0.05, init_kappa=10000.0),
]
SCENARIO_ORDER = [scenario["scenario"] for scenario in INIT_SCENARIOS]
NUM_WORKERS = min(8, os.cpu_count() or 1)
JOBLIB_BACKEND = "loky"

runs = pd.DataFrame([
    {**BASE_SPEC, **scenario, "algo": algo, "seed": seed}
    for scenario in INIT_SCENARIOS
    for algo in ALGOS
    for seed in SEEDS
])
runs["problem"] = "MatrixFactorization"
runs.insert(0, "run_id", range(len(runs)))
runs["scenario"] = pd.Categorical(runs["scenario"], categories=SCENARIO_ORDER, ordered=True)
runs["scale_ratio"] = runs[["left_scale", "right_scale"]].max(axis=1) / runs[["left_scale", "right_scale"]].min(axis=1)
preview_cols = [
    "run_id", "scenario", "problem", "algo", "d", "rank", "factor_rank", "seed", "lr", "iters",
    "early_stop", "early_stop_min_steps", "early_stop_patience", "early_stop_min_delta",
    "spectrum", "kappa", "init_mode", "left_scale", "right_scale", "scale_ratio", "init_kappa",
    "device_type", "dtype_name",
]
runs = runs[preview_cols]
runs["scenario"] = runs["scenario"].astype(str)

print(f"device={DEVICE}, workers={NUM_WORKERS}, backend={JOBLIB_BACKEND}")
print(f"smoke_test={SMOKE_TEST}, smoke_test_max_steps={SMOKE_TEST_MAX_STEPS}")
print(f"scenarios={len(INIT_SCENARIOS)}, runs={len(runs)}, max_iters={BASE_SPEC['iters']}, max_total_steps={len(runs) * BASE_SPEC['iters']}")
print(
    "early_stop=patience on absolute loss improvement "
    f"(min_steps={BASE_SPEC['early_stop_min_steps']}, "
    f"patience={BASE_SPEC['early_stop_patience']}, "
    f"min_delta={BASE_SPEC['early_stop_min_delta']})"
)
IPython.display.display(pd.DataFrame(INIT_SCENARIOS))
IPython.display.display(runs)


## Experiment Pseudocode

```text
for each initialization scenario:
    for each algorithm and seed:
        build X_star
        initialize L and R according to the scenario
        record initial_loss
        choose optimizer
        for step in range(iters):
            loss = 0.5 * mean((L @ R.T - X_star)^2)
            backprop through L and R
            optimizer.step()
            record loss, grad_norm, elapsed time
            stop if patience-based early stopping fires
concat all per-run tables back into runs
```

This is longer than E02 because it multiplies the optimizer/seed grid by multiple initialization scenarios.

## Worker Definition

In [ ]:
def dtype_from_name(dtype_name):
    dtype = getattr(torch, dtype_name)
    if not isinstance(dtype, torch.dtype):
        raise ValueError(f"unknown torch dtype: {dtype_name}")
    return dtype


def configure_torch(dtype):
    torch.set_default_dtype(dtype)
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass


def make_optimizer(algo, params, lr, rank):
    if algo == "Muon" and hasattr(torch.optim, "Muon"):
        return torch.optim.Muon(params, lr=lr, weight_decay=0.0, momentum=0.9, nesterov=False, ns_steps=5)
    if algo == "Muon":
        return optimizers.MuonExact(params, lr=lr, momentum=0.9, variant="newton_schulz", ns_steps=5)
    if algo in {"Muon-Exact", "MuonExact"}:
        return optimizers.MuonExact(params, lr=lr, momentum=0.9, variant="exact")
    if algo == "Shampoo":
        return optimizers.Shampoo(params, lr=lr, beta2=0.9, epsilon=1e-8)
    if algo == "Adam":
        return torch.optim.Adam(params, lr=lr)
    if algo == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"unknown algo: {algo}")


def column_scales(width, scale, init_kappa, device, dtype, *, reverse=False):
    if width <= 1 or float(init_kappa) <= 1.0:
        return torch.full((width,), float(scale), device=device, dtype=dtype)
    exponents = torch.linspace(0.0, 1.0, width, device=device, dtype=dtype)
    values = float(scale) * torch.pow(torch.tensor(float(init_kappa), device=device, dtype=dtype), -exponents)
    return torch.flip(values, dims=(0,)) if reverse else values


def initialize_factors(run, d, factor_rank, seed, device, dtype):
    mode = run["init_mode"]
    left = problems.MatrixConstruction.randn((d, factor_rank), seed + 3000, device, dtype)
    right = problems.MatrixConstruction.randn((d, factor_rank), seed + 4000, device, dtype)

    if mode == "gaussian":
        return left * float(run["left_scale"]), right * float(run["right_scale"])
    if mode == "column-ill-conditioned":
        left_scales = column_scales(factor_rank, run["left_scale"], run["init_kappa"], device, dtype)
        right_scales = column_scales(factor_rank, run["right_scale"], run["init_kappa"], device, dtype)
        return left * left_scales, right * right_scales
    if mode == "opposite-column-ill-conditioned":
        left_scales = column_scales(factor_rank, run["left_scale"], run["init_kappa"], device, dtype)
        right_scales = column_scales(factor_rank, run["right_scale"], run["init_kappa"], device, dtype, reverse=True)
        return left * left_scales, right * right_scales
    raise ValueError(f"unknown init_mode: {mode}")


def single_run(run):
    run = dict(run)
    d = int(run["d"])
    rank = int(run["rank"])
    factor_rank = int(run["factor_rank"])
    seed = int(run["seed"])
    iters = int(run["iters"])
    early_stop = bool(run["early_stop"])
    early_stop_min_steps = int(run["early_stop_min_steps"])
    early_stop_patience = int(run["early_stop_patience"])
    early_stop_min_delta = float(run["early_stop_min_delta"])
    device = torch.device(run["device_type"])
    dtype = dtype_from_name(run["dtype_name"])
    configure_torch(dtype)

    def initialize_problem(run):
        return problems.MatrixFactorization.make_matrix_factorization_problem(
            d,
            rank,
            spectrum=run["spectrum"],
            kappa=float(run["kappa"]),
            seed=seed,
            device=device,
            dtype=dtype,
            factor_rank=factor_rank,
        )

    def optimization(problem, run):
        left_init, right_init = initialize_factors(run, d, factor_rank, seed, device, dtype)
        left = torch.nn.Parameter(left_init)
        right = torch.nn.Parameter(right_init)
        params = [left, right]
        state = {
            "problem": problem,
            "left": left,
            "right": right,
            "params": params,
            "optimizer": make_optimizer(run["algo"], params, float(run["lr"]), rank),
            "step": 0,
            "initial_loss": float(problem.loss(left, right).detach().cpu()),
            "best_loss": None,
            "early_stop_wait": 0,
        }

        def step(state):
            state["optimizer"].zero_grad(set_to_none=True)
            loss = state["problem"].loss(state["left"], state["right"])
            loss.backward()
            grad_norms = torch.stack([param.grad.detach().norm() for param in state["params"]])
            state["grad_norm"] = float(torch.linalg.vector_norm(grad_norms).cpu())
            state["optimizer"].step()
            state["step"] += 1
            state["loss"] = float(loss.detach().cpu())

        def update_early_stop_state(state):
            if state["best_loss"] is None:
                state["best_loss"] = state["loss"]
                return
            absolute_improvement = state["best_loss"] - state["loss"]
            if absolute_improvement >= early_stop_min_delta:
                state["best_loss"] = state["loss"]
                state["early_stop_wait"] = 0
            else:
                state["early_stop_wait"] += 1

        def should_stop(state):
            return (
                early_stop
                and state["step"] >= early_stop_min_steps
                and state["early_stop_wait"] >= early_stop_patience
            )

        def stepwise_data(state, start_time, stop_reason):
            return {
                **run,
                "step": state["step"],
                "initial_loss": state["initial_loss"],
                "loss": state["loss"],
                "grad_norm": state["grad_norm"],
                "best_loss": state["best_loss"],
                "early_stop_wait": state["early_stop_wait"],
                "elapsed_s": time.perf_counter() - start_time,
                "stop_reason": stop_reason,
            }

        rows = []
        start_time = time.perf_counter()
        for _ in range(iters):
            step(state)
            update_early_stop_state(state)
            stop_reason = "early_stop_patience" if should_stop(state) else ""
            rows.append(stepwise_data(state, start_time, stop_reason))
            if stop_reason:
                break
        return pd.DataFrame(rows)

    return optimization(initialize_problem(run), run)


## Plot Setup

In [ ]:
for module_name in list(sys.modules):
    if module_name == "plotting" or module_name.startswith("plotting."):
        sys.modules.pop(module_name)

import plotting


def show_figure(fig):
    IPython.display.display(fig)
    plt.close(fig)


## Execute

This cell runs every row in `runs` through `single_run`. After it runs, `runs` becomes the long per-step result table.

In [ ]:
runs = util.run_experiments(
    runs,
    single_run,
    num_workers=NUM_WORKERS,
    backend=JOBLIB_BACKEND,
    algo_order=ALGOS,
    sort_columns=("scenario", "algo", "d", "seed", "step"),
)
runs["scenario"] = pd.Categorical(runs["scenario"], categories=SCENARIO_ORDER, ordered=True)
runs["algo"] = pd.Categorical(runs["algo"], categories=ALGOS, ordered=True)
runs = runs.sort_values(["scenario", "algo", "seed", "step"]).reset_index(drop=True)
runs["scenario"] = runs["scenario"].astype(str)
runs["algo"] = runs["algo"].astype(str)
IPython.display.display(runs)


## Result Tables

In [ ]:
ordered_runs = runs.sort_values(["scenario", "run_id", "step"])
run_summary = ordered_runs.groupby("run_id", as_index=False).first()
run_summary = run_summary.drop(columns=["loss", "grad_norm", "best_loss", "early_stop_wait", "elapsed_s", "stop_reason"])
run_summary = run_summary.merge(
    ordered_runs.groupby("run_id", as_index=False).agg(
        final_loss=("loss", "last"),
        min_loss=("loss", "min"),
        actual_steps=("step", "max"),
        time_s=("elapsed_s", "max"),
        stop_reason=("stop_reason", "last"),
    ),
    on="run_id",
)
run_summary["stopped_early"] = run_summary["actual_steps"] < run_summary["iters"]
run_summary["scenario"] = pd.Categorical(run_summary["scenario"], categories=SCENARIO_ORDER, ordered=True)
run_summary["algo"] = pd.Categorical(run_summary["algo"], categories=ALGOS, ordered=True)
run_summary = run_summary.sort_values(["scenario", "algo", "seed"]).reset_index(drop=True)
run_summary["scenario"] = run_summary["scenario"].astype(str)
run_summary["algo"] = run_summary["algo"].astype(str)

scenario_summary = run_summary.groupby(["scenario", "algo"], as_index=False).agg(
    runs=("run_id", "count"),
    initial_loss_mean=("initial_loss", "mean"),
    min_loss_mean=("min_loss", "mean"),
    final_loss_mean=("final_loss", "mean"),
    actual_steps_mean=("actual_steps", "mean"),
    time_s_mean=("time_s", "mean"),
    stopped_early_rate=("stopped_early", "mean"),
)
scenario_summary["scenario"] = pd.Categorical(scenario_summary["scenario"], categories=SCENARIO_ORDER, ordered=True)
scenario_summary["algo"] = pd.Categorical(scenario_summary["algo"], categories=ALGOS, ordered=True)
scenario_summary = scenario_summary.sort_values(["scenario", "algo"]).reset_index(drop=True)
scenario_summary["scenario"] = scenario_summary["scenario"].astype(str)
scenario_summary["algo"] = scenario_summary["algo"].astype(str)


def trajectories_from(frame):
    ordered = frame.sort_values(["run_id", "step"])
    return {
        (str(algo), int(d), int(seed)): {
            "loss": group["loss"].tolist(),
            "grad_norm": group["grad_norm"].tolist(),
        }
        for (algo, d, seed), group in ordered.groupby(["algo", "d", "seed"], sort=False)
    }

IPython.display.display(runs)
IPython.display.display(run_summary)
IPython.display.display(scenario_summary)


## Scenario Initial Loss

In [ ]:
fig, ax = plotting.plot_scenario_metric(run_summary, "initial_loss", "Mean initial loss across initialization scenarios", "initial loss")
show_figure(fig)

fig, ax = plotting.plot_scenario_metric(run_summary, "initial_loss", "Mean initial loss across initialization scenarios", "initial loss", log_y=True)
show_figure(fig)


## Scenario Optimization Metrics

In [ ]:
fig, ax = plotting.plot_scenario_metric(run_summary, "actual_steps", "Mean executed steps across initialization scenarios", "steps")
show_figure(fig)

fig, ax = plotting.plot_scenario_metric(run_summary, "time_s", "Mean wall-clock across initialization scenarios", "seconds")
show_figure(fig)


## Scenario Minimum Loss

In [ ]:
fig, ax = plotting.plot_scenario_metric(run_summary, "min_loss", "Mean minimum loss across initialization scenarios", "loss")
show_figure(fig)

fig, ax = plotting.plot_scenario_metric(run_summary, "min_loss", "Mean minimum loss across initialization scenarios", "loss", log_y=True)
show_figure(fig)


## Scenario Final Loss

In [ ]:
fig, ax = plotting.plot_scenario_metric(run_summary, "final_loss", "Mean final loss across initialization scenarios", "loss")
show_figure(fig)

fig, ax = plotting.plot_scenario_metric(run_summary, "final_loss", "Mean final loss across initialization scenarios", "loss", log_y=True)
show_figure(fig)


## Per-Scenario Metric Overview

In [ ]:
for scenario in SCENARIO_ORDER:
    sub_summary = run_summary[run_summary["scenario"] == scenario]
    for loss_log_y in (False, True):
        fig, axes = plotting.plot_metric_overview(sub_summary, loss_log_y=loss_log_y)
        fig.suptitle(f"{scenario}: metric overview", y=1.02)
        show_figure(fig)


## Per-Scenario Algorithm Trajectories

In [ ]:
for scenario in SCENARIO_ORDER:
    sub_runs = runs[runs["scenario"] == scenario]
    scenario_trajectories = trajectories_from(sub_runs)
    for log_y in (False, True):
        fig, ax = plotting.plot_algorithms_for_dimension(scenario_trajectories, BASE_SPEC["d"], log_y=log_y)
        fig.suptitle(scenario, y=1.02)
        show_figure(fig)


## Per-Scenario Seed Variability

In [ ]:
for scenario in SCENARIO_ORDER:
    sub_runs = runs[runs["scenario"] == scenario]
    scenario_trajectories = trajectories_from(sub_runs)
    for log_y in (False, True):
        fig, ax = plotting.plot_seed_variability_for_dimension(scenario_trajectories, BASE_SPEC["d"], log_y=log_y)
        fig.suptitle(scenario, y=1.02)
        show_figure(fig)


## Conclusion

In [ ]:
def conclusion_markdown(run_summary):
    max_steps = int(run_summary["iters"].sum())
    actual_steps = int(run_summary["actual_steps"].sum())
    stopped = int(run_summary["stopped_early"].sum())
    lines = [
        "### Result Summary",
        "",
        f"- Initialization scenarios: `{len(SCENARIO_ORDER)}`",
        f"- Runs: `{len(run_summary)}`",
        f"- Methods: `{', '.join(sorted(run_summary['algo'].unique()))}`",
        f"- Max iterations per run: `{int(run_summary['iters'].iloc[0])}`",
        f"- Max optimizer-step budget: `{max_steps}`",
        f"- Executed optimizer steps: `{actual_steps}` (`{max_steps - actual_steps}` skipped by early stopping)",
        f"- Early-stopped runs: `{stopped}/{len(run_summary)}`",
        "",
    ]
    for scenario in SCENARIO_ORDER:
        sub = scenario_summary[scenario_summary["scenario"] == scenario]
        easiest_start = sub.loc[sub["initial_loss_mean"].idxmin()]
        fewest_steps = sub.loc[sub["actual_steps_mean"].idxmin()]
        best_loss = sub.loc[sub["min_loss_mean"].idxmin()]
        lines.append(
            f"- {scenario}: lowest mean initial_loss is `{easiest_start.algo}` at `{easiest_start.initial_loss_mean:.3e}`; "
            f"fewest executed steps is `{fewest_steps.algo}` at `{fewest_steps.actual_steps_mean:.1f}`; "
            f"lowest min_loss is `{best_loss.algo}` at `{best_loss.min_loss_mean:.3e}`."
        )
    lines.append("")
    lines.append("Tiny balanced initialization exposes the flat-region issue: loss can be nontrivial while gradients are small because each factor gradient is multiplied by the other factor.")
    lines.append("Unbalanced and column-ill-conditioned initialization test whether an optimizer can correct badly scaled factor coordinates, not just reduce the objective from an isotropic start.")
    return "\n".join(lines)


IPython.display.display(IPython.display.Markdown(conclusion_markdown(run_summary)))
